## Load Comments from YouTube

In [3]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

def get_comments(fileName, index):
    data = pd.read_csv(f"raw/{fileName}.csv", sep=";")
    data = data["Comment"]
    return data.to_csv(f"only-comment/comment-{index}.csv", index=False)

In [4]:
get_comments("youtube-comments", "1")
get_comments("youtube-comments-2", "2")
get_comments("youtube-comments-3", "3")
get_comments("youtube-comments-4", "4")
get_comments("youtube-comments-5", "5")
get_comments("youtube-comments-6", "6")
get_comments("youtube-comments-7", "7")
get_comments("youtube-comments-8", "8")
get_comments("youtube-comments-9", "9")
get_comments("youtube-comments-10", "10")

## Normalize Words in `judol_keywords.txt`

In [43]:
import unicodedata
import re 
from unidecode import unidecode
import os

In [46]:
# ==========================================
# 1. KONFIGURASI NORMALISASI (THE ENGINE)
# ==========================================

def normalize_all_cases(text):
    """
    Menangani berbagai teknik adversarial text:
    - Unicode Font (𝐉𝐀𝐋𝐀𝐊 -> JALAK)
    - Zalgo/Coretan (C̶̨̨... -> CAMAR)
    - Simbol Lingkaran (🅚 -> K)
    - Superscript (ᴶᵁᴺᴼ -> JUNO)
    - Spasi antar huruf (G A C O R -> GACOR)
    """
    if not isinstance(text, str): return ""
    
    # Mapping visual manual untuk karakter yang sering salah diartikan unidecode
    visual_map = {
        'Δ': 'a', 'ά': 'a', 'ᗩ': 'a', '@': 'a',
        'ᗯ': 'w', 'vv': 'w',
        'σ': 'o', '𝐎': 'o', '0': 'o',
        '𝐍': 'n', 'И': 'n',
        '𝓽': 't', '𝓣': 't',
        '𝐒': 's', '🅢': 's',
        '|': 'i', '!': 'i',
    }
    
    for char, replacement in visual_map.items():
        text = text.replace(char, replacement)
    
    # Transliterasi simbol/lingkaran ke ASCII standar
    text = unidecode(text)
    
    # Normalisasi NFKD untuk memisahkan 'combining marks' (coretan/zalgo)
    text = unicodedata.normalize('NFKD', text)
    text = "".join([c for c in text if unicodedata.category(c) != 'Mn'])
    
    # Lowercase & Hapus karakter non-alfanumerik (kecuali spasi)
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # Aggressive Space Collapse (Menghapus spasi di antara huruf tunggal)
    # Loop memastikan "k u r i r" -> "kurir" secara tuntas
    while True:
        new_text = re.sub(r'(?<=\b[a-z])\s+(?=[a-z]\b)', '', text)
        if new_text == text: break
        text = new_text
        
    return " ".join(text.split()).strip()

# ==========================================
# 2. PREPARASI KATA KUNCI (KEYWORD CLEANING)
# ==========================================

def prepare_keywords(input_path):
    print(f"[*] Menormalisasi daftar kata kunci dari {input_path}...")
    with open(input_path, 'r', encoding='utf-8') as f:
        raw_keywords = f.readlines()
    
    clean_keywords = set()
    for kw in raw_keywords:
        normalized = normalize_all_cases(kw.strip())
        if normalized:
            clean_keywords.add(normalized)
            
    # Simpan versi bersih untuk referensi debugging
    output_path = input_path.replace(".txt", "_clean.txt")
    with open(output_path, 'w', encoding='utf-8') as f:
        for kw in sorted(list(clean_keywords)):
            f.write(kw + '\n')
            
    print(f"[+] Berhasil! {len(clean_keywords)} keyword unik siap digunakan.")
    return list(clean_keywords)

# ==========================================
# 3. PROSES PELABELAN MASIF
# ==========================================

def run_mass_labelling():
    # Load & Clean Keywords
    kw_path = "judol_keywords.txt"
    keywords = prepare_keywords(kw_path)
    
    # Path folder
    base_folder = "only-comment/"
    
    for i in range(1, 11):
        file_name = f"comment-{i}.csv"
        file_path = os.path.join(base_folder, file_name)
        
        if os.path.exists(file_path):
            print(f"\n[>] Memproses {file_name}...")
            df = pd.read_csv(file_path)
            
            # 1. Normalisasi Teks Komentar
            df['comment_cleaned'] = df['Comment'].apply(normalize_all_cases)
            
            # 2. Pelabelan (Substring Matching)
            # Melabeli 1 jika ada satu saja keyword yang terkandung dalam teks bersih
            df['label'] = df['comment_cleaned'].apply(
                lambda x: 1 if any(kw in x for kw in keywords) else 0
            )
            
            # 3. Simpan Hasil
            output_name = f"labeled-{file_name}"
            df.to_csv(os.path.join(base_folder, output_name), index=False)
            
            # Summary singkat per file
            total_judi = df['label'].sum()
            print(f"    - Selesai! {total_judi} iklan terdeteksi.")
        else:
            print(f"\n[!] File {file_name} tidak ditemukan, melewati...")

if __name__ == "__main__":
    run_mass_labelling()

[*] Menormalisasi daftar kata kunci dari judol_keywords.txt...
[+] Berhasil! 166 keyword unik siap digunakan.

[>] Memproses comment-1.csv...
    - Selesai! 85 iklan terdeteksi.

[>] Memproses comment-2.csv...
    - Selesai! 40 iklan terdeteksi.

[>] Memproses comment-3.csv...
    - Selesai! 316 iklan terdeteksi.

[>] Memproses comment-4.csv...
    - Selesai! 25 iklan terdeteksi.

[>] Memproses comment-5.csv...
    - Selesai! 66 iklan terdeteksi.

[>] Memproses comment-6.csv...
    - Selesai! 130 iklan terdeteksi.

[>] Memproses comment-7.csv...
    - Selesai! 27 iklan terdeteksi.

[>] Memproses comment-8.csv...
    - Selesai! 9 iklan terdeteksi.

[>] Memproses comment-9.csv...
    - Selesai! 360 iklan terdeteksi.

[>] Memproses comment-10.csv...
    - Selesai! 1 iklan terdeteksi.


In [48]:
def collect_gambling_ads(base_folder, output_file):
    all_ads = []
    
    print("[*] Memulai pengumpulan data Label 1...")
    
    for i in range(1, 11):
        file_name = f"labeled-comment-{i}.csv"
        file_path = os.path.join(base_folder, file_name)
        
        if os.path.exists(file_path):
            # Membaca file hasil pelabelan sebelumnya
            df = pd.read_csv(file_path)
            
            # Filter hanya yang memiliki label 1
            ads_only = df[df['label'] == 1][['Comment', 'label']]
            
            # Ubah nama kolom 'Comment' menjadi 'komentar' agar konsisten dengan dataset sebelumnya
            ads_only = ads_only.rename(columns={'Comment': 'komentar'})
            
            # Tambahkan ke list
            all_ads.append(ads_only)
            print(f"    - Berhasil mengambil {len(ads_only)} baris dari {file_name}")
        else:
            print(f"    - [!] File {file_name} tidak ditemukan.")

    if all_ads:
        # Gabungkan semua DataFrame dalam list
        final_df = pd.concat(all_ads, axis=0, ignore_index=True)
        
        # Simpan ke CSV
        final_df.to_csv(output_file, index=False)
        print(f"\n[DONE] Total {len(final_df)} data iklan judi berhasil dikumpulkan!")
        print(f"[DONE] File disimpan di: {output_file}")
    else:
        print("\n[!] Tidak ada data label 1 yang ditemukan.")

# --- EKSEKUSI ---
folder_path = "only-comment/"
output_path = "only-comment/all_judol_ads.csv"

collect_gambling_ads(folder_path, output_path)

[*] Memulai pengumpulan data Label 1...
    - Berhasil mengambil 85 baris dari labeled-comment-1.csv
    - Berhasil mengambil 40 baris dari labeled-comment-2.csv
    - Berhasil mengambil 316 baris dari labeled-comment-3.csv
    - Berhasil mengambil 25 baris dari labeled-comment-4.csv
    - Berhasil mengambil 66 baris dari labeled-comment-5.csv
    - Berhasil mengambil 130 baris dari labeled-comment-6.csv
    - Berhasil mengambil 27 baris dari labeled-comment-7.csv
    - Berhasil mengambil 9 baris dari labeled-comment-8.csv
    - Berhasil mengambil 360 baris dari labeled-comment-9.csv
    - Berhasil mengambil 1 baris dari labeled-comment-10.csv

[DONE] Total 1059 data iklan judi berhasil dikumpulkan!
[DONE] File disimpan di: only-comment/all_judol_ads.csv
